# Oxta — Cascade Validation (CHRASS sweep + TTT + VIB + Slender + K·m + FULL)

**Estrutura cascata em etapas:**

1. **Baseline** (tudo OFF) — referência absoluta
2. **CHRASS density sweep** — 10, 20, 30, 40, 50, 60, 70% → identifica melhor density
3. **Outras tech individuais** — TTT, Pantheon VIB (β=0.01), Slender, Nemotron K·m
4. **FULL integrado** = CHRASS(best) + TTT + VIB + K·m  (sem Slender)
5. **FULL + Slender** = todos juntos
6. **OXB throughput** — benchmark BitPacker (separado, não treina modelo)
7. **Análise final** — tabela comparativa + plot 14 curves overlaid

**Smoke 10M (default):** 6L × d=256 × 4H × vocab=8192. STEPS=1500.  ~15min/run em T4 → ~3.5h total.
**Pra 40M (mais robusto, ~6h):** mude `LAYERS=12 D_MODEL=384` na Cell 4.

**Pré-requisito:** push do código pra branch `oxta-contabil` antes de abrir o Colab (notebook faz git clone).

In [ ]:
!nvidia-smi | head -15
!pip install --quiet pybind11 numpy zstandard matplotlib

In [ ]:
# Git clone do repo (mesmo padrão dos outros Colabs Oxta)
import os, subprocess, sys
from pathlib import Path

REPO_OWNER = 'vitorGgC569'
REPO_NAME = 'reimagined'
BRANCH = 'oxta-contabil'
REPO_ROOT = Path('/content/reimagined-main')

clone_url = f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git'
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    if token:
        clone_url = f'https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git'
        print('[auth] GITHUB_TOKEN OK')
except Exception:
    pass

if not REPO_ROOT.exists():
    subprocess.run(['git', 'clone', '--depth=1', '--branch', BRANCH,
                    clone_url, str(REPO_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'remote', 'set-url',
                    'origin', clone_url], check=False)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'reset', '--hard',
                    f'origin/{BRANCH}'], check=True)

os.environ['REPO_ROOT'] = str(REPO_ROOT)
os.chdir(REPO_ROOT)
subprocess.run(['git', 'log', '--oneline', '-1'], check=True)

In [ ]:
# Build CUDA-enabled nsos_ext for T4 (sm_75) — ~10 min na primeira execução
BUILD_DIR = REPO_ROOT / 'OXN/nsos/build-colab'
BUILD_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run(['cmake',
    '-S', str(REPO_ROOT / 'OXN/nsos'),
    '-B', str(BUILD_DIR),
    '-DNSOS_ENABLE_CUDA=ON',
    '-DNSOS_BUILD_PYTHON=ON',
    '-DNSOS_BUILD_TESTS=OFF',
    '-DNSOS_BUILD_CLI=OFF',
    '-DNSOS_BUILD_API=OFF',
    '-DNSOS_BUILD_OXTAMEM=OFF',
    '-DNSOS_CUDA_ARCHITECTURES=75',
], check=True)
subprocess.run(['cmake', '--build', str(BUILD_DIR),
    '--target', 'nsos_ext', '-j'], check=True)
print('Build OK')

In [ ]:
# Validate import + confirma que TODAS as flags da cascata estão expostas
sys.path.insert(0, str(BUILD_DIR))
import nsos_ext
cfg = nsos_ext.ModelConfig()
needed = ['use_chrass', 'chrass_density', 'use_ttt',
          'pantheon_vib_beta', 'use_slender_embedding',
          'moe_expert_hidden_dim']
missing = [f for f in needed if not hasattr(cfg, f)]
if missing:
    raise RuntimeError(f'Flags ausentes na binding (re-push branch): {missing}')
print('All flags exposed:', needed)

## Knobs da cascata

In [ ]:
# === KNOBS ===
LAYERS = 6         # 6L × d=256 ≈ 10M;  12L × d=384 ≈ 40M (mais robusto, mais lento)
D_MODEL = 256
N_HEADS = 4
VOCAB = 8192
STEPS = 1500
SEED = 42
OUT_DIR = '/content/cascade_reports'

# CHRASS sweep — densidades a testar (10-70% passo 10%)
CHRASS_DENSITIES = [10, 20, 30, 40, 50, 60, 70]
# Pra mais rápido (só 4 pontos): CHRASS_DENSITIES = [10, 30, 50, 70]

!mkdir -p {OUT_DIR}

PROBE = str(REPO_ROOT / 'OXN/nsos/scripts/probe_cascade.py')
ANALYZE = str(REPO_ROOT / 'OXN/nsos/scripts/analyze_cascade.py')
PYTHONPATH_ENV = str(BUILD_DIR)

import time
def run_probe(label, *args, skip_if_exists=True):
    out_path = Path(OUT_DIR) / f'probe_{label}_s{SEED}.json'
    if skip_if_exists and out_path.exists():
        print(f'\n[SKIP] {label} (already exists at {out_path})')
        return
    print(f'\n{"="*60}\n  RUN: {label}\n{"="*60}')
    t0 = time.time()
    cmd = ['python', PROBE,
           '--label', label,
           '--seed', str(SEED),
           '--steps', str(STEPS),
           '--layers', str(LAYERS),
           '--d-model', str(D_MODEL),
           '--n-heads', str(N_HEADS),
           '--vocab-size', str(VOCAB),
           '--out-dir', OUT_DIR, '--cuda'] + list(args)
    env = {**os.environ, 'PYTHONPATH': PYTHONPATH_ENV}
    p = subprocess.run(cmd, env=env, capture_output=True, text=True)
    print(p.stdout[-2200:])
    if p.returncode != 0:
        print('STDERR:', p.stderr[-1200:])
    print(f'  done in {(time.time()-t0)/60:.1f} min')

## Etapa 1 — Baseline (referência)

In [ ]:
# Run 1: baseline (tudo OFF)
run_probe('baseline')

## Etapa 2 — CHRASS density SWEEP (10-70%)

7 runs em loop, cada um igual ao baseline mas com CHRASS na density indicada.
**~1h45 nesta etapa**. No fim, célula identifica a melhor density com base na loss final.

In [ ]:
for d in CHRASS_DENSITIES:
    label = f'chrass{d:02d}'
    run_probe(label, '--use-chrass', '--chrass-density', str(d))

In [ ]:
# Mid-cascade: identifica a MELHOR density do sweep CHRASS
import json, glob, statistics
reports = {}
for f in glob.glob(f'{OUT_DIR}/probe_*.json'):
    r = json.loads(open(f).read())
    reports[r['label']] = r

base_w = reports['baseline']['loss']['all'][-200:]
base_final = statistics.fmean(base_w)
print(f'Baseline final-200 loss: {base_final:.4f}\n')
print(f'{"density":<10} {"final":>8} {"delta_vs_base":>14} {"verdict":>22}')
print('-' * 60)
best_density, best_delta = None, float('inf')
for d in CHRASS_DENSITIES:
    lbl = f'chrass{d:02d}'
    if lbl not in reports: continue
    w = reports[lbl]['loss']['all'][-200:]
    f = statistics.fmean(w)
    delta = (f - base_final) / max(abs(base_final), 1e-6) * 100
    if delta < best_delta:
        best_delta, best_density = delta, d
    v = 'HELP' if delta < -1 else ('HURT' if delta > 1 else 'neutral')
    print(f'{d:>4}%      {f:>8.4f} {delta:>+13.1f}% {v:>22}')
print()
print(f'>>> BEST density: {best_density}% (delta {best_delta:+.1f}% vs baseline)')
if best_delta > 1:
    print('!! TODAS as densities atrapalham. FULL runs usarão CHRASS desligado.')
    BEST_CHRASS_DENSITY = 0
else:
    BEST_CHRASS_DENSITY = best_density

## Etapa 3 — Outras tecnologias individuais

In [ ]:
# Run: TTT alone
run_probe('ttt', '--use-ttt')

In [ ]:
# Run: Pantheon VIB alone (beta=0.01)
run_probe('vib001', '--vib-beta', '0.01')

In [ ]:
# Run: Slender embedding alone
run_probe('slender', '--use-slender')

In [ ]:
# Run: Nemotron K·m alone (m=5120 + MoE on)
run_probe('km5120', '--use-moe', '--moe-expert-hidden', '5120')

## Etapa 4 — FULL integrado (usa BEST_CHRASS_DENSITY do sweep)

In [ ]:
# FULL = CHRASS(best) + TTT + VIB + K·m   (sem Slender)
full_args = ['--use-ttt', '--vib-beta', '0.01',
             '--use-moe', '--moe-expert-hidden', '5120']
if BEST_CHRASS_DENSITY > 0:
    full_args = ['--use-chrass', '--chrass-density', str(BEST_CHRASS_DENSITY)] + full_args
run_probe(f'full_d{BEST_CHRASS_DENSITY}', *full_args)

In [ ]:
# FULL + Slender = todos juntos
run_probe(f'full_slender_d{BEST_CHRASS_DENSITY}', *full_args, '--use-slender')

## Etapa 5 — OXB throughput (só I/O, não treina modelo)

Demonstra a velocidade do BitPacker isolada do treino.

In [ ]:
# Build OXB aion_core Python module (~1 min)
OXB_BUILD = REPO_ROOT / 'OXB/aion_core_cpp/build-colab'
subprocess.run(['cmake',
    '-S', str(REPO_ROOT / 'OXB/aion_core_cpp'),
    '-B', str(OXB_BUILD)], check=True)
subprocess.run(['cmake', '--build', str(OXB_BUILD),
    '--config', 'Release', '--target', 'aion_core', '-j'], check=True)

In [ ]:
sys.path.insert(0, str(OXB_BUILD / 'Release'))
sys.path.insert(0, str(OXB_BUILD))
import aion_core as ac
import numpy as np

n, vocab, bits = 10_000_000, 8192, 13
np.random.seed(42)
toks = np.random.randint(0, vocab, size=n, dtype=np.uint32)
t0 = time.perf_counter()
packed = ac.BitPacker.pack(toks, bits)
t = time.perf_counter() - t0
print(f'OXB BitPacker: {n:,} tokens in {t*1000:.0f} ms ({n/t/1e6:.0f} M tok/s)')
print(f'  size: {n*4/1024/1024:.1f} MB raw -> {len(packed)*8/1024/1024:.1f} MB packed ({len(packed)*8/(n*4)*100:.0f}%)')

## Etapa 6 — Análise final cascata + plot

In [ ]:
subprocess.run(['python', ANALYZE,
    '--reports-dir', OUT_DIR,
    '--plot'], check=True)

In [ ]:
from IPython.display import Image, display
display(Image(f'{OUT_DIR}/cascade_curves.png'))

In [ ]:
# Tabela do veredito
import json, statistics, glob
reports = {}
for f in glob.glob(f'{OUT_DIR}/probe_*.json'):
    r = json.loads(open(f).read())
    reports[r['label']] = r

base = reports.get('baseline')
if base:
    bf = statistics.fmean(base['loss']['all'][-200:])
    print(f'\n{"label":<22} {"final":>8} {"delta":>8} {"ms/step":>10} {"verdict":>22}')
    print('-' * 75)
    for lbl in sorted(reports.keys()):
        r = reports[lbl]
        f = statistics.fmean(r['loss']['all'][-200:])
        m = r['timing']['ms_per_step']
        delta = (f - bf) / abs(bf) * 100
        if lbl == 'baseline':
            print(f'{lbl:<22} {f:>8.4f} {"ref":>8} {m:>10.1f} {"reference":>22}')
            continue
        if delta < -2:   v = '** MEANINGFUL HELP'
        elif delta < -0.5: v = '+ small help'
        elif delta > 2:  v = '!! MEANINGFUL HURT'
        elif delta > 0.5: v = '- small hurt'
        else:            v = '= neutral'
        print(f'{lbl:<22} {f:>8.4f} {delta:>+7.1f}% {m:>10.1f} {v:>22}')

In [ ]:
# Zipa reports pra download
!cd /content && zip -r cascade_reports.zip cascade_reports/
from google.colab import files
files.download('/content/cascade_reports.zip')